In [ ]:
import tensorflow as tf
import numpy as np

from dataclasses import dataclass, fields
from tensorflow.keras import layers
from pathlib import Path
from PIL import Image

# Data

In [ ]:
def load_images_from_dir():
    ds_path = Path("/kaggle/input/datasets/spandan2/cats-faces-64x64-for-generative-models/cats")

    raw_images = np.stack([
        np.array(Image.open(f).convert("RGB").resize((64, 64)))
        for f in ds_path.iterdir()
        if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ])

    return raw_images

In [ ]:
def preprocess_image(raw_image):
    """casts uint8 [0, 255] -> float32 [-1.0, 1.0]."""

    image = tf.cast(raw_image, tf.float32)
    image = (image / 127.5) - 1.0
    image.set_shape([64, 64, 3])

    return image

In [ ]:
def build_train_dataset(raw_images, global_batch_size):
    train_dataset = (
        tf.data.Dataset.from_tensor_slices(raw_images)
        .shuffle(buffer_size=raw_images.shape[0], reshuffle_each_iteration=True)
        .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(
            global_batch_size,
            drop_remainder=True,
        )
        .repeat()
        .prefetch(tf.data.AUTOTUNE)
    )

    return train_dataset

In [ ]:
def build_dist_train_dataset(strategy, global_batch_size):
    raw_images = load_images_from_dir()
    N_SAMPLES = raw_images.shape[0]

    dataset = build_train_dataset(raw_images, global_batch_size)
    dist_dataset = strategy.experimental_distribute_dataset(dataset)

    return dist_dataset, N_SAMPLES

# Misc

In [ ]:
@dataclass(frozen=True)
class Paths:
    checkpoint_dir: Path = Path('checkpoints')
    generated_images_dir: Path = Path('generated_images')


for field in fields(Paths):
    getattr(Paths, field.name).mkdir(exist_ok=True)

In [ ]:
def get_strategy(strategy_str: str = 'gpu') -> tf.distribute.Strategy:
    strategy_str = strategy_str.lower()

    if strategy_str == 'tpu':
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)

        return tf.distribute.TPUStrategy(resolver)

    return tf.distribute.MirroredStrategy()

# Models

In [ ]:
# generator for 64x64x3 images
def build_generator(latent_dim=128):
    inputs = layers.Input(shape=(latent_dim,))

    # 4x4 resolution
    x = layers.Dense(4 * 4 * 512, use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((4, 4, 512))(x)

    # 4x4 -> 8x8
    x = layers.Conv2DTranspose(256, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # 8x8 -> 16x16
    x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # 16x16 -> 32x32
    x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # 32x32 -> 64x64
    x = layers.Conv2DTranspose(32, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # output: (64, 64, 3) normalized in [-1, 1]
    outputs = layers.Conv2D(3, kernel_size=3, padding="same", activation="tanh")(x)

    return tf.keras.Model(inputs, outputs, name="generator")

In [ ]:
def build_critic(image_shape=(64, 64, 3)):
    # note: no BatchNormalization in WGAN-GP critic
    inputs = layers.Input(shape=image_shape)

    # 64x64 -> 32x32
    x = layers.Conv2D(64, kernel_size=4, strides=2, padding="same")(inputs)
    x = layers.LeakyReLU(0.2)(x)

    # 32x32 -> 16x16
    x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    # 16x16 -> 8x8
    x = layers.Conv2D(256, kernel_size=4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    # 8x8 -> 4x4
    x = layers.Conv2D(512, kernel_size=4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Flatten()(x)
    outputs = layers.Dense(1)(x)

    return tf.keras.Model(inputs, outputs, name="critic")

In [ ]:
class WGANGP:
    def __init__(
        self,
        generator: tf.keras.Model,
        critic: tf.keras.Model,
        generator_optimizer: tf.keras.Optimizer,
        critic_optimizer: tf.keras.Optimizer,
        per_replica_batch_size: int,
        global_batch_size: int,
        latent_dim: int = 128,
        gp_weight: float = 10.0,
    ):
        self.generator = generator
        self.critic = critic

        self.g_optimizer = generator_optimizer
        self.c_optimizer = critic_optimizer

        self.latent_dim = latent_dim
        self.gp_weight = gp_weight

        self.global_batch_size = global_batch_size
        self.per_replica_batch_size = per_replica_batch_size

    def _gradient_penalty(self, real_images, fake_images):
        alpha = tf.random.uniform([self.per_replica_batch_size, 1, 1, 1], 0.0, 1.0, dtype=real_images.dtype)
        interpolated = real_images + alpha * (fake_images - real_images)

        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = self.critic(interpolated, training=True)

        grads = gp_tape.gradient(pred, interpolated)

        # epsilon 1e-12 prevents sqrt(0) -> NaN gradients
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-12)
        gp_per_sample = tf.square(norm - 1.0)

        return gp_per_sample

    def critic_step(self, real_images):
        noise = tf.random.normal([self.per_replica_batch_size, self.latent_dim], dtype=real_images.dtype)

        with tf.GradientTape() as tape:
            fake_images = self.generator(noise, training=True)
            fake_logits = self.critic(fake_images, training=True)
            real_logits = self.critic(real_images, training=True)

            # per_sample
            wasserstein_loss_per_sample = fake_logits - real_logits
            gp_per_sample = self._gradient_penalty(real_images, fake_images)
            critic_loss_per_sample = wasserstein_loss_per_sample + self.gp_weight * tf.reshape(gp_per_sample, [-1, 1])

            # distributed-safe
            wasserstein_distance = tf.nn.compute_average_loss(-wasserstein_loss_per_sample, global_batch_size=self.global_batch_size)
            gp_loss = tf.nn.compute_average_loss(gp_per_sample, global_batch_size=self.global_batch_size)
            critic_loss = tf.nn.compute_average_loss(critic_loss_per_sample, global_batch_size=self.global_batch_size)

            # equivalent to
            # c_loss = (tf.reduce_sum(fake_logits) - tf.reduce_sum(real_logits) + (self.gp_weight * tf.reduce_sum(gp))) * (1.0 / self.global_batch_size)

        critic_grads = tape.gradient(critic_loss, self.critic.trainable_variables)
        self.c_optimizer.apply_gradients(zip(critic_grads, self.critic.trainable_variables))

        return {
            "wasserstein_distance": wasserstein_distance,
            "gp_loss": gp_loss,
            "critic_loss": critic_loss,
        }

    def generator_step(self):
        noise = tf.random.normal([self.per_replica_batch_size, self.latent_dim])

        with tf.GradientTape() as tape:
            fake_images = self.generator(noise, training=True)
            fake_logits = self.critic(fake_images, training=True)

            gen_loss_per_sample = -fake_logits
            gen_loss = tf.nn.compute_average_loss(gen_loss_per_sample, global_batch_size=self.global_batch_size)

            # equivalent to
            # gen_loss = -tf.reduce_sum(fake_logits) * (1.0 / self.global_batch_size)

        gen_grads = tape.gradient(gen_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(gen_grads, self.generator.trainable_variables))

        return {
            "generator_loss": gen_loss
        }

In [ ]:
class DistributedWGANGPTrainer:
    def __init__(
        self,
        strategy: tf.distribute.Strategy,
        wgan: WGANGP,
        # monitor: GANMonitor,
        per_replica_batch_size: int,
        global_batch_size: int,
        n_critic_steps: int = 5,
    ):
        self.strategy = strategy

        self.wgan = wgan
        # self.monitor = monitor

        self.n_critic_steps = n_critic_steps

        self.per_replica_batch_size = per_replica_batch_size
        self.global_batch_size = global_batch_size

    def _reduce_per_replica_metrics(self, metrics: dict[str, tf.Tensor]) -> dict[str, tf.Tensor]:
        return {k: self.strategy.reduce(tf.distribute.ReduceOp.SUM, v, axis=None) for k, v in metrics.items()}

    @tf.function
    def _distributed_critic_step(self, real_images) -> dict[str, tf.Tensor]:
        per_replica_metrics = self.strategy.run(self.wgan.critic_step, args=(real_images,))
        metrics = self._reduce_per_replica_metrics(per_replica_metrics)

        return metrics

    @tf.function
    def _distributed_generator_step(self) -> dict[str, tf.Tensor]:
        per_replica_metrics = self.strategy.run(self.wgan.generator_step)
        metrics = self._reduce_per_replica_metrics(per_replica_metrics)

        return metrics

    def train(self, distributed_dataset, total_samples: int, epochs: int):
        data_iter = iter(distributed_dataset)

        total_batches = total_samples // self.global_batch_size
        steps_per_epoch = total_batches // self.n_critic_steps

        for epoch in range(epochs):
            for step in range(steps_per_epoch):
                for _ in range(self.n_critic_steps):
                    real_images = next(data_iter)
                    critic_metrics = self._distributed_critic_step(real_images)

                gen_metrics = self._distributed_generator_step()

                metrics = critic_metrics | gen_metrics

                if step % 5 == 0:
                    print(
                        f"Epoch [{epoch + 1}/{epochs}] "
                        f"Step [{step + 1}/{steps_per_epoch}] "
                        + " | ".join(f"{k}: {v:.4f}" for k, v in metrics.items())
                    )

# Training

In [ ]:
strategy = get_strategy('gpu')
print(f"Number of synchronized devices: {strategy.num_replicas_in_sync}")

In [ ]:
PER_REPLICA_BATCH_SIZE = 64
GLOBAL_BATCH_SIZE = strategy.num_replicas_in_sync * PER_REPLICA_BATCH_SIZE

LATENT_DIM = 128
GP_WEIGHT = 10.0
N_CRITIC_STEPS = 5

EPOCHS = 5

In [ ]:
dist_dataset, N_SAMPLES = build_dist_train_dataset(strategy, GLOBAL_BATCH_SIZE)

In [ ]:
with strategy.scope():
    generator = build_generator(latent_dim=LATENT_DIM)
    critic = build_critic(image_shape=(64, 64, 3))

    generator_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.0, beta_2=0.9)
    critic_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.0, beta_2=0.9)

    # pre-allocates variables across all replicas in eager mode
    # this prevents lazy tf.cond initialization inside the graph
    generator_optimizer.build(generator.trainable_variables)
    critic_optimizer.build(critic.trainable_variables)

    wgan = WGANGP(
        generator=generator,
        critic=critic,
        generator_optimizer=generator_optimizer,
        critic_optimizer=critic_optimizer,
        per_replica_batch_size=PER_REPLICA_BATCH_SIZE,
        global_batch_size=GLOBAL_BATCH_SIZE,
        latent_dim=LATENT_DIM,
        gp_weight=GP_WEIGHT,
    )

trainer = DistributedWGANGPTrainer(
    strategy=strategy,
    wgan=wgan,
    # monitor: GANMonitor,
    per_replica_batch_size=PER_REPLICA_BATCH_SIZE,
    global_batch_size=GLOBAL_BATCH_SIZE,
    n_critic_steps=N_CRITIC_STEPS,
)

print("Starting multi-GPU training...")
trainer.train(dist_dataset, total_samples=N_SAMPLES, epochs=EPOCHS)